# Drift Protocol DEX Trading Bot

Trades **BTC-PERP** on [Drift Protocol](https://drift.trade) (Solana) using the same
signal engine as the CEX bot: EMA crossover + ADX + ATR + optional 3-TF confluence.

**Execution flow:**
```
Signal engine (unchanged)
      ↓
Decides LONG / SHORT / flat
      ↓
DriftExecutor builds & sends transaction
      ↓
Phantom keypair signs on-chain
      ↓
Trade executed on Drift perpetuals market
      ↓
PnL tracked from on-chain position
```

**Modes:** `backtest` | `paper` | `live`  
Start with `backtest`, then `paper`, then `live` only when satisfied.

**Required `.env` file** (same folder as this notebook):
```
SOLANA_PRIVATE_KEY=your_base58_private_key_exported_from_phantom
HELIUS_RPC_URL=https://mainnet.helius-rpc.com/?api-key=your_key
```

> **Never share your private key or commit the `.env` file to git.**

## 1) Install Dependencies

In [ ]:
# Uncomment and run once to install all required packages
# !pip install -q pandas numpy ccxt python-dotenv
# !pip install -q solders anchorpy
# !pip install -q driftpy
#
# driftpy requires Python >= 3.10 and a relatively recent pip.
# If driftpy install fails try: pip install driftpy --pre

## 2) Configuration

All settings live here. The only new ones vs the CEX bot are the **Solana / Drift** block.
Everything else (timeframes, indicators, risk, HTF2 filter) is identical.

In [ ]:
import os
import logging
from dotenv import load_dotenv
load_dotenv()  # loads .env file from current directory

# ── Mode ─────────────────────────────────────────────────────────────────────
# backtest  → run on historical CCXT data, no wallet needed
# paper     → live signals, simulated fills, no wallet needed
# live      → real on-chain execution via Drift — USE WITH CARE
MODE = os.getenv("MODE", "backtest").lower()

# ── Solana / Drift ────────────────────────────────────────────────────────────
# SOLANA_PRIVATE_KEY : base58 private key exported from Phantom
# HELIUS_RPC_URL     : your Helius (or QuickNode) mainnet RPC endpoint
#
# Drift market options for DRIFT_MARKET_INDEX:
#   0 = SOL-PERP  |  1 = BTC-PERP  |  2 = ETH-PERP
SOLANA_PRIVATE_KEY  = os.getenv("SOLANA_PRIVATE_KEY", "")
HELIUS_RPC_URL      = os.getenv("HELIUS_RPC_URL", "")
DRIFT_MARKET_INDEX  = int(os.getenv("DRIFT_MARKET_INDEX", "1"))   # 1 = BTC-PERP
DRIFT_LEVERAGE      = float(os.getenv("DRIFT_LEVERAGE", "1.0"))   # 1x = no leverage
DRIFT_ENV           = os.getenv("DRIFT_ENV", "mainnet")           # mainnet | devnet

# ── Price data source for signals ─────────────────────────────────────────────
# Signals are generated from CCXT market data (Binance public endpoint).
# The wallet / Drift settings above are only used for execution.
SIGNAL_BROKER = os.getenv("SIGNAL_BROKER", "binance").lower()
SYMBOL        = os.getenv("SYMBOL", "BTC/USDT")   # CCXT symbol for price data

# ── Timeframes ────────────────────────────────────────────────────────────────
LTF = os.getenv("LTF", "5m")   # entry timeframe
HTF = os.getenv("HTF", "1h")   # first trend filter

# ── HTF2 (second trend filter — 3-TF confluence) ─────────────────────────────
# True  → LONG  only when HTF2 bull + HTF bull + LTF cross up
#         SHORT only when HTF2 bear + HTF bear + LTF cross down
# False → original 2-TF logic
USE_HTF2        = os.getenv("USE_HTF2", "true").lower() == "true"
HTF2            = os.getenv("HTF2", "4h")
HTF2_BARS       = int(os.getenv("HTF2_BARS", "500"))
PAPER_HTF2_DAYS = int(os.getenv("PAPER_HTF2_DAYS", "120"))

# ── Lookback ──────────────────────────────────────────────────────────────────
DATA_PULL_MODE  = os.getenv("DATA_PULL_MODE", "days").lower()  # days | bars
LTF_BARS        = int(os.getenv("LTF_BARS", "1500"))
HTF_BARS        = int(os.getenv("HTF_BARS", "1500"))
BACKTEST_DAYS   = int(os.getenv("BACKTEST_DAYS", "30"))
PAPER_LTF_DAYS  = int(os.getenv("PAPER_LTF_DAYS", "30"))
PAPER_HTF_DAYS  = int(os.getenv("PAPER_HTF_DAYS", "60"))

# ── Strategy params ───────────────────────────────────────────────────────────
EMA_FAST      = int(os.getenv("EMA_FAST", "9"))
EMA_SLOW      = int(os.getenv("EMA_SLOW", "21"))
ADX_PERIOD    = int(os.getenv("ADX_PERIOD", "14"))
ADX_THRESHOLD = float(os.getenv("ADX_THRESHOLD", "25"))
ATR_PERIOD    = int(os.getenv("ATR_PERIOD", "14"))
ATR_SL_MULT   = float(os.getenv("ATR_SL_MULT", "1.8"))
ATR_TP_MULT   = float(os.getenv("ATR_TP_MULT", "2.5"))

# ── Risk & capital ────────────────────────────────────────────────────────────
# RISK_PCT        : % of available collateral to risk per trade
# INITIAL_CAPITAL : used for backtest/paper simulation only
#                   in live mode the bot reads your actual Drift USDC balance
RISK_PCT         = float(os.getenv("RISK_PCT", "5.0"))
INITIAL_CAPITAL  = float(os.getenv("INITIAL_CAPITAL", "100"))

# ── Paper / live loop ─────────────────────────────────────────────────────────
PAPER_POLL_INTERVAL = int(os.getenv("PAPER_POLL_INTERVAL", "60"))  # seconds
PAPER_MAX_TRADES    = int(os.getenv("PAPER_MAX_TRADES", "0"))       # 0 = unlimited

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger("drift_bot")

# ── Startup summary ───────────────────────────────────────────────────────────
htf2_display = f" / {HTF2} (ON)" if USE_HTF2 else " (HTF2 OFF)"
wallet_display = SOLANA_PRIVATE_KEY[:6] + "..." if SOLANA_PRIVATE_KEY else "NOT SET"
rpc_display    = HELIUS_RPC_URL[:40] + "..." if HELIUS_RPC_URL else "NOT SET"
print(
    "Drift Bot Config:\n"
    f"  Mode           : {MODE}\n"
    f"  Drift env      : {DRIFT_ENV}  Market index: {DRIFT_MARKET_INDEX} (BTC-PERP)  Leverage: {DRIFT_LEVERAGE}x\n"
    f"  Wallet         : {wallet_display}\n"
    f"  RPC            : {rpc_display}\n"
    f"  Signal source  : {SIGNAL_BROKER.upper()}  Symbol: {SYMBOL}\n"
    f"  LTF/HTF/HTF2   : {LTF} / {HTF}{htf2_display}\n"
    f"  Lookback mode  : {DATA_PULL_MODE}  BACKTEST_DAYS={BACKTEST_DAYS}\n"
    f"  Capital        : ${INITIAL_CAPITAL:,.2f}  Risk/Trade: {RISK_PCT}%  (live: reads Drift balance)\n"
)

if MODE == 'live':
    if not SOLANA_PRIVATE_KEY:
        raise EnvironmentError("SOLANA_PRIVATE_KEY is not set. Add it to your .env file.")
    if not HELIUS_RPC_URL:
        raise EnvironmentError("HELIUS_RPC_URL is not set. Add it to your .env file.")
    print("LIVE MODE — real transactions will be submitted to Drift Protocol.")
    print("Make sure you have USDC deposited on Drift before proceeding.\n")

## 3) Data Fetchers

Unchanged from the CEX bot. Price data always comes from Binance public OHLCV
(no API key required). The Drift wallet is only used for order execution.

In [ ]:
import time
from datetime import datetime
import numpy as np
import pandas as pd

_TF_MIN = {"1m":1,"3m":3,"5m":5,"15m":15,"30m":30,"1h":60,"2h":120,"4h":240,"6h":360,"12h":720,"1d":1440}

def _ccxt_client(name: str, key: str = '', secret: str = ''):
    try:
        import ccxt
    except Exception as e:
        raise RuntimeError("ccxt is required: pip install ccxt") from e
    opts = {
        'enableRateLimit': True,
        'options': {'defaultType': 'spot', 'adjustForTimeDifference': True},
        'timeout': 20000,
    }
    if key:
        opts.update({'apiKey': key or None, 'secret': secret or None})
    return getattr(ccxt, name)(opts)

def _fetch_ccxt_ohlcv_paginated(ex, symbol: str, timeframe: str, candles: int) -> pd.DataFrame:
    per_call = 1000
    out, since, remaining = [], None, int(candles)
    while remaining > 0:
        batch = min(per_call, remaining)
        ohlcv = ex.fetch_ohlcv(symbol, timeframe, since=since, limit=batch)
        if not ohlcv:
            break
        out.extend(ohlcv)
        remaining -= len(ohlcv)
        since = ohlcv[-1][0] + 1
        rl = getattr(ex, 'rateLimit', None)
        if rl:
            time.sleep(rl / 1000.0)
        if len(ohlcv) < batch:
            break
    df = pd.DataFrame(out, columns=["timestamp","open","high","low","close","volume"]).astype(
        {"open":float,"high":float,"low":float,"close":float,"volume":float}
    )
    if df.empty:
        return df
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df.set_index("timestamp").sort_index()

def fetch_demo_data(symbol: str, timeframe: str, *, days: int = 365) -> pd.DataFrame:
    mins = _TF_MIN.get(timeframe, 60)
    n    = max(300, int(days * 1440 / mins))
    rng  = pd.date_range(end=datetime.now(), periods=n, freq=f"{mins}min")
    np.random.seed(42)
    lr = np.random.normal(0.0001, 0.002, n)
    block = max(1, n // 6)
    for i in range(0, n, block):
        lr[i:i+block] += np.random.choice([-1, 1]) * 0.0003
    prices = 100 * np.exp(np.cumsum(lr))
    high   = prices * (1 + np.abs(np.random.normal(0, 0.003, n)))
    low    = prices * (1 - np.abs(np.random.normal(0, 0.003, n)))
    open_  = np.roll(prices, 1); open_[0] = prices[0]
    vol    = np.random.randint(1000, 50000, n).astype(float)
    return pd.DataFrame({"open":open_,"high":high,"low":low,"close":prices,"volume":vol}, index=rng)

def get_data(symbol: str, timeframe: str, *, bars: int = None, days: int = None) -> pd.DataFrame:
    source = SIGNAL_BROKER.upper()
    try:
        ex     = _ccxt_client(SIGNAL_BROKER)
        mins   = _TF_MIN.get(timeframe, 60)
        n_bars = bars or max(500, int((days or 30) * 1440 / mins))
        df     = _fetch_ccxt_ohlcv_paginated(ex, symbol, timeframe, n_bars)
    except Exception as e:
        log.error(f"Data fetch failed ({SIGNAL_BROKER}): {e} — using DEMO data.")
        df     = fetch_demo_data(symbol, timeframe, days=days or 30)
        source = 'DEMO'
    if len(df):
        span = (df.index[-1] - df.index[0]).days
        print(f"Data pull {symbol:<12} {timeframe:<4} src:{source:<8} "
              f"candles:{len(df):>5}  span:{span}d "
              f"({df.index[0].strftime('%Y-%m-%d')} -> {df.index[-1].strftime('%Y-%m-%d')})")
    else:
        print("Data pull returned 0 candles.")
    return df

## 4) Indicators

Identical to the CEX bot — EMA, ATR, ADX.

In [ ]:
import numpy as np
import pandas as pd

def ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()

def calc_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    hl = df['high'] - df['low']
    hc = (df['high'] - df['close'].shift()).abs()
    lc = (df['low']  - df['close'].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

def calc_adx(df: pd.DataFrame, period: int = 14) -> pd.Series:
    up   = df['high'].diff()
    down = -df['low'].diff()
    pdm  = np.where((up > down) & (up > 0), up, 0.0)
    mdm  = np.where((down > up) & (down > 0), down, 0.0)
    tr   = calc_atr(df, period)
    pdi  = 100 * pd.Series(pdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    mdi  = 100 * pd.Series(mdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    dx   = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean()

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['ema_fast'] = ema(df['close'], EMA_FAST)
    df['ema_slow'] = ema(df['close'], EMA_SLOW)
    df['atr']      = calc_atr(df, ATR_PERIOD)
    df['atr_ma']   = df['atr'].rolling(ATR_PERIOD * 2).mean()
    df['adx']      = calc_adx(df, ADX_PERIOD)
    return df.dropna()

## 5) Signal Generation

Identical to the CEX bot — 2-TF or 3-TF confluence, EMA cross + ADX + ATR filters.

In [ ]:
import numpy as np
import pandas as pd

def generate_signals(ltf_df: pd.DataFrame, htf_df: pd.DataFrame,
                     htf2_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    LONG  : HTF2 bull (if ON) + HTF bull + LTF EMA cross up   + ADX > threshold + ATR expanding
    SHORT : HTF2 bear (if ON) + HTF bear + LTF EMA cross down + ADX > threshold + ATR expanding
    """
    df = ltf_df.copy()
    htf_trend = (htf_df['ema_fast'] > htf_df['ema_slow']).astype(int).reindex(df.index, method='ffill')

    if USE_HTF2 and htf2_df is not None:
        htf2_trend = (htf2_df['ema_fast'] > htf2_df['ema_slow']).astype(int).reindex(df.index, method='ffill')
    else:
        htf2_trend = None

    cross_up   = (df['ema_fast'] > df['ema_slow']) & (df['ema_fast'].shift() <= df['ema_slow'].shift())
    cross_down = (df['ema_fast'] < df['ema_slow']) & (df['ema_fast'].shift() >= df['ema_slow'].shift())
    trend_ok   = df['adx'] > ADX_THRESHOLD
    vol_ok     = df['atr'] > df['atr_ma']

    df['signal'] = 0
    if USE_HTF2 and htf2_trend is not None:
        df.loc[cross_up   & trend_ok & vol_ok & (htf_trend == 1) & (htf2_trend == 1), 'signal'] =  1
        df.loc[cross_down & trend_ok & vol_ok & (htf_trend == 0) & (htf2_trend == 0), 'signal'] = -1
    else:
        df.loc[cross_up   & trend_ok & vol_ok & (htf_trend == 1), 'signal'] =  1
        df.loc[cross_down & trend_ok & vol_ok & (htf_trend == 0), 'signal'] = -1

    sign = df['signal']
    df['sl'] = np.where(sign ==  1, df['close'] - ATR_SL_MULT * df['atr'],
                np.where(sign == -1, df['close'] + ATR_SL_MULT * df['atr'], np.nan))
    df['tp'] = np.where(sign ==  1, df['close'] + ATR_TP_MULT * df['atr'],
                np.where(sign == -1, df['close'] - ATR_TP_MULT * df['atr'], np.nan))
    return df

## 6) Backtester

Unchanged from the CEX bot. Used to validate signal quality before going live.

In [ ]:
import pandas as pd
import numpy as np

class Backtester:
    def __init__(self, df: pd.DataFrame):
        self.df           = df
        self.equity       = INITIAL_CAPITAL
        self.trades       = []
        self.equity_curve = []

    def run(self):
        pos = entry_price = sl = tp = entry_time = None
        for ts, row in self.df.iterrows():
            self.equity_curve.append({'timestamp': ts, 'equity': self.equity})
            if pos is not None:
                hit_sl = (pos ==  1 and row['low']  <= sl) or (pos == -1 and row['high'] >= sl)
                hit_tp = (pos ==  1 and row['high'] >= tp) or (pos == -1 and row['low']  <= tp)
                if hit_sl or hit_tp:
                    exit_price = sl if hit_sl else tp
                    pct  = (exit_price - entry_price) / entry_price * pos
                    rb   = ATR_SL_MULT * row['atr'] / max(entry_price, 1e-9)
                    pnl  = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
                    self.equity += pnl
                    self.trades.append({
                        'entry_time': entry_time, 'exit_time': ts,
                        'direction': 'LONG' if pos == 1 else 'SHORT',
                        'entry_price': entry_price, 'exit_price': exit_price,
                        'sl': sl, 'tp': tp, 'pnl': pnl,
                        'result': 'WIN' if hit_tp else 'LOSS'
                    })
                    pos = None
            if pos is None and row['signal'] != 0:
                pos         = row['signal']
                entry_price = row['close']
                entry_time  = ts
                sl          = row['sl']
                tp          = row['tp']
        return self._stats()

    def _stats(self):
        if not self.trades:
            return {'error': 'No trades — try lowering ADX_THRESHOLD or extending lookback'}
        df      = pd.DataFrame(self.trades)
        wins    = df[df['result'] == 'WIN']
        losses  = df[df['result'] == 'LOSS']
        eq      = pd.DataFrame(self.equity_curve).set_index('timestamp')
        dd      = (eq['equity'] - eq['equity'].cummax()) / eq['equity'].cummax() * 100
        ret     = df['pnl'] / INITIAL_CAPITAL
        sh      = (ret.mean() / ret.std() * np.sqrt(252)) if ret.std() > 0 else 0
        return {
            'total_trades' : int(len(df)),
            'win_rate'     : round(len(wins) / len(df) * 100, 2) if len(df) else 0.0,
            'total_return' : round((self.equity - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100, 2),
            'final_equity' : round(self.equity, 2),
            'sharpe_ratio' : round(sh, 2),
            'max_drawdown' : round(float(dd.min()) if not dd.empty else 0.0, 2),
            'avg_win'      : round(float(wins['pnl'].mean())   if len(wins)   else 0.0, 2),
            'avg_loss'     : round(float(losses['pnl'].mean()) if len(losses) else 0.0, 2),
            'profit_factor': round(
                float(wins['pnl'].sum() / abs(losses['pnl'].sum()))
                if len(losses) and abs(losses['pnl'].sum()) > 0 else float('inf'), 2),
            'trades_df'    : df,
            'equity_curve' : eq,
        }

## 7) Drift Executor

This is the new layer — handles all Solana / Drift interaction.

**What it does:**
- Loads your keypair from the private key in `.env`
- Connects to Drift via your Helius RPC
- Opens and closes perpetual positions (market orders)
- Reads your live USDC collateral balance for position sizing
- Places SL/TP as separate reduce-only orders after entry

**In `backtest` / `paper` mode** this class is never instantiated — no wallet needed.

In [ ]:
# ── DriftExecutor ─────────────────────────────────────────────────────────────
# Only imported / instantiated when MODE == 'live'

class DriftExecutor:
    """
    Wraps driftpy to open/close BTC-PERP positions on Drift Protocol.

    Usage:
        executor = DriftExecutor()          # connects on init
        await executor.open_long(sl, tp)    # open long, place SL+TP
        await executor.open_short(sl, tp)   # open short, place SL+TP
        await executor.close_position()     # flatten any open position
        bal = await executor.get_balance()  # USDC collateral in dollars
    """

    def __init__(self):
        # Deferred imports so the notebook runs without driftpy in backtest/paper mode
        try:
            from solders.keypair import Keypair
            from anchorpy import Provider, Wallet
            from solana.rpc.async_api import AsyncClient
            from driftpy.drift_client import DriftClient
            from driftpy.constants.numeric_constants import BASE_PRECISION, PRICE_PRECISION
            from driftpy.types import (
                MarketType, OrderType, OrderParams,
                PositionDirection, TriggerCondition
            )
            from driftpy.account_subscription_config import AccountSubscriptionConfig
        except ImportError as e:
            raise ImportError(
                "driftpy / solders / anchorpy are required for live mode.\n"
                "Install with: pip install driftpy solders anchorpy"
            ) from e

        import base58, asyncio

        # Store references
        self._MarketType         = MarketType
        self._OrderType          = OrderType
        self._OrderParams        = OrderParams
        self._PositionDirection  = PositionDirection
        self._TriggerCondition   = TriggerCondition
        self._BASE_PRECISION     = BASE_PRECISION
        self._PRICE_PRECISION    = PRICE_PRECISION

        # Build keypair from base58 private key
        secret_bytes = base58.b58decode(SOLANA_PRIVATE_KEY)
        keypair      = Keypair.from_bytes(secret_bytes)
        wallet       = Wallet(keypair)

        # Async Solana RPC client
        self._rpc_client = AsyncClient(HELIUS_RPC_URL)
        provider         = Provider(self._rpc_client, wallet)

        # Drift client
        env = "mainnet" if DRIFT_ENV == "mainnet" else "devnet"
        self._drift = DriftClient(
            provider,
            account_subscription=AccountSubscriptionConfig("cached"),
            env=env,
        )
        self._market_index = DRIFT_MARKET_INDEX
        self._loop         = asyncio.get_event_loop()
        log.info("DriftExecutor: subscribing to Drift account data...")
        self._loop.run_until_complete(self._drift.subscribe())
        log.info("DriftExecutor: ready.")

    # ── Public helpers ────────────────────────────────────────────────────────

    def get_balance(self) -> float:
        """Return available USDC collateral in dollars."""
        return self._loop.run_until_complete(self._async_get_balance())

    def open_long(self, sl_price: float, tp_price: float) -> str:
        """Open a long position sized by RISK_PCT of collateral, then place SL+TP."""
        return self._loop.run_until_complete(self._async_open(1, sl_price, tp_price))

    def open_short(self, sl_price: float, tp_price: float) -> str:
        """Open a short position sized by RISK_PCT of collateral, then place SL+TP."""
        return self._loop.run_until_complete(self._async_open(-1, sl_price, tp_price))

    def close_position(self) -> str:
        """Market-close any open position on this market."""
        return self._loop.run_until_complete(self._async_close())

    def get_open_position(self):
        """Return the current Drift PerpPosition or None."""
        return self._loop.run_until_complete(self._async_get_position())

    # ── Async internals ───────────────────────────────────────────────────────

    async def _async_get_balance(self) -> float:
        user = self._drift.get_user()
        # free_collateral returns value in USDC with PRICE_PRECISION decimals
        collateral = user.get_free_collateral() / self._PRICE_PRECISION
        return float(collateral)

    async def _async_open(self, direction: int, sl_price: float, tp_price: float) -> str:
        """
        Size the position using RISK_PCT of available collateral, then
        submit a market order followed by SL and TP trigger orders.
        """
        PD   = self._PositionDirection
        OT   = self._OrderType
        MT   = self._MarketType
        OP   = self._OrderParams
        TC   = self._TriggerCondition
        BP   = self._BASE_PRECISION
        PP   = self._PRICE_PRECISION

        is_long    = direction == 1
        drift_dir  = PD.Long() if is_long else PD.Short()

        # Position sizing: risk RISK_PCT% of collateral
        collateral    = await self._async_get_balance()
        risk_dollars  = collateral * RISK_PCT / 100.0
        # Get current oracle price from Drift
        oracle_price  = self._drift.get_oracle_price_data_and_slot(
            MT.Perp(), self._market_index
        ).data.price / PP
        # Base size = risk_dollars / oracle_price * leverage
        base_size     = risk_dollars / oracle_price * DRIFT_LEVERAGE
        base_amt      = int(base_size * BP)   # scaled to BASE_PRECISION

        log.info(
            f"Opening {'LONG' if is_long else 'SHORT'}  "
            f"oracle={oracle_price:.2f}  size={base_size:.6f} BTC  "
            f"risk=${risk_dollars:.2f}  collateral=${collateral:.2f}"
        )

        # ── Market entry order ────────────────────────────────────────────────
        entry_params = OP(
            order_type        = OT.Market(),
            market_type       = MT.Perp(),
            direction         = drift_dir,
            base_asset_amount = base_amt,
            market_index      = self._market_index,
        )
        tx_entry = await self._drift.place_perp_order(entry_params)
        log.info(f"Entry order submitted: {tx_entry}")

        # ── Stop-loss (trigger, reduce-only) ──────────────────────────────────
        sl_trigger  = TC.Below() if is_long else TC.Above()
        sl_dir      = PD.Short() if is_long else PD.Long()  # opposite direction to close
        sl_params = OP(
            order_type         = OT.TriggerMarket(),
            market_type        = MT.Perp(),
            direction          = sl_dir,
            base_asset_amount  = base_amt,
            market_index       = self._market_index,
            trigger_price      = int(sl_price * PP),
            trigger_condition  = sl_trigger,
            reduce_only        = True,
        )
        tx_sl = await self._drift.place_perp_order(sl_params)
        log.info(f"SL order submitted  @ {sl_price:.2f}: {tx_sl}")

        # ── Take-profit (trigger, reduce-only) ───────────────────────────────
        tp_trigger  = TC.Above() if is_long else TC.Below()
        tp_params = OP(
            order_type         = OT.TriggerMarket(),
            market_type        = MT.Perp(),
            direction          = sl_dir,   # same as SL — closes the position
            base_asset_amount  = base_amt,
            market_index       = self._market_index,
            trigger_price      = int(tp_price * PP),
            trigger_condition  = tp_trigger,
            reduce_only        = True,
        )
        tx_tp = await self._drift.place_perp_order(tp_params)
        log.info(f"TP order submitted  @ {tp_price:.2f}: {tx_tp}")

        return tx_entry

    async def _async_close(self) -> str:
        """Cancel all open orders on this market then market-close the position."""
        await self._drift.cancel_orders(market_index=self._market_index,
                                         market_type=self._MarketType.Perp())
        tx = await self._drift.close_position(self._market_index)
        log.info(f"Position closed: {tx}")
        return tx

    async def _async_get_position(self):
        user = self._drift.get_user()
        try:
            return user.get_perp_position(self._market_index)
        except Exception:
            return None

    def shutdown(self):
        """Unsubscribe and close the RPC connection cleanly."""
        self._loop.run_until_complete(self._drift.unsubscribe())
        self._loop.run_until_complete(self._rpc_client.close())
        log.info("DriftExecutor: disconnected.")

## 8) Live Trading Loop

Replaces the `PaperTrader` for live mode. Polls for new candles on every
`PAPER_POLL_INTERVAL` seconds, evaluates all three timeframe filters, and
sends orders to Drift when a signal fires.

**State machine:**
```
FLAT → signal fires → open position (entry + SL + TP orders on-chain)
IN_POSITION → SL or TP hit on-chain → Drift closes automatically
             → bot detects flat → ready for next signal
```

In [ ]:
import time

class PaperTrader:
    """Simulated fills — no wallet needed. Identical to CEX bot paper trader."""

    def __init__(self):
        self.equity      = INITIAL_CAPITAL
        self.position    = None
        self.entry_price = self.entry_time = self.sl = self.tp = None
        self.trade_log   = []
        self.iteration   = 0

    def _unrealised_pct(self, price):
        if self.position == 'LONG':  return (price - self.entry_price) / self.entry_price * 100
        if self.position == 'SHORT': return (self.entry_price - price) / self.entry_price * 100
        return 0.0

    def _record_exit(self, ts, row, exit_price, hit_tp):
        sign = 1 if self.position == 'LONG' else -1
        pct  = (exit_price - self.entry_price) / self.entry_price * sign
        rb   = ATR_SL_MULT * row['atr'] / max(self.entry_price, 1e-9)
        pnl  = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
        self.equity += pnl
        self.trade_log.append({
            'entry_time': self.entry_time, 'exit_time': ts,
            'direction': self.position,
            'entry_price': self.entry_price, 'exit_price': exit_price,
            'sl': self.sl, 'tp': self.tp, 'pnl': pnl,
            'result': 'WIN' if hit_tp else 'LOSS'
        })
        self.position = self.entry_price = self.entry_time = self.sl = self.tp = None

    def _fetch_all(self):
        if DATA_PULL_MODE == 'bars':
            ltf_raw  = get_data(SYMBOL, LTF,  bars=LTF_BARS)
            htf_raw  = get_data(SYMBOL, HTF,  bars=HTF_BARS)
            htf2_raw = get_data(SYMBOL, HTF2, bars=HTF2_BARS) if USE_HTF2 else None
        else:
            ltf_raw  = get_data(SYMBOL, LTF,  days=PAPER_LTF_DAYS)
            htf_raw  = get_data(SYMBOL, HTF,  days=PAPER_HTF_DAYS)
            htf2_raw = get_data(SYMBOL, HTF2, days=PAPER_HTF2_DAYS) if USE_HTF2 else None
        return ltf_raw, htf_raw, htf2_raw

    def run(self):
        poll = PAPER_POLL_INTERVAL
        htf2_info = f" + {HTF2}" if USE_HTF2 else ""
        print(f"Paper trader — {SYMBOL} {LTF}/{HTF}{htf2_info}  "
              f"Capital: ${INITIAL_CAPITAL:,.2f}  Poll: {poll}s  Ctrl+C to stop\n")
        while True:
            try:
                self.iteration += 1
                ltf_raw, htf_raw, htf2_raw = self._fetch_all()
                ltf  = compute_indicators(ltf_raw)
                htf  = compute_indicators(htf_raw)
                htf2 = compute_indicators(htf2_raw) if USE_HTF2 and htf2_raw is not None else None

                row, prev = ltf.iloc[-1], ltf.iloc[-2]
                ts        = ltf.index[-1]
                htf_bull  = htf.iloc[-1]['ema_fast'] > htf.iloc[-1]['ema_slow']
                htf2_bull = (htf2.iloc[-1]['ema_fast'] > htf2.iloc[-1]['ema_slow']) if htf2 is not None else None

                cross_up   = (row['ema_fast'] > row['ema_slow']) and (prev['ema_fast'] <= prev['ema_slow'])
                cross_down = (row['ema_fast'] < row['ema_slow']) and (prev['ema_fast'] >= prev['ema_slow'])
                adx_ok     = row['adx'] > ADX_THRESHOLD
                atr_ok     = row['atr'] > row['atr_ma']

                if USE_HTF2 and htf2_bull is not None:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull  and htf2_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull) and (not htf2_bull)
                else:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull)

                if self.position:
                    hit_sl = (self.position=='LONG'  and row['low']  <= self.sl) or \
                             (self.position=='SHORT' and row['high'] >= self.sl)
                    hit_tp = (self.position=='LONG'  and row['high'] >= self.tp) or \
                             (self.position=='SHORT' and row['low']  <= self.tp)
                    if hit_sl or hit_tp:
                        self._record_exit(ts, row, self.sl if hit_sl else self.tp, hit_tp)
                        self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)
                        if PAPER_MAX_TRADES and len(self.trade_log) >= PAPER_MAX_TRADES:
                            print(f"Reached max trades ({PAPER_MAX_TRADES}). Stopping.")
                            break
                        time.sleep(poll); continue
                    self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)
                    time.sleep(poll); continue

                if long_ok or short_ok:
                    self.position    = 'LONG' if long_ok else 'SHORT'
                    self.entry_price = row['close']
                    self.entry_time  = ts
                    sgn = 1 if long_ok else -1
                    self.sl = row['close'] - sgn * ATR_SL_MULT * row['atr']
                    self.tp = row['close'] + sgn * ATR_TP_MULT * row['atr']
                self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)
            except KeyboardInterrupt:
                print("\nStopped by user.")
                break
            except Exception as e:
                log.error(f"Loop error: {e}")
            time.sleep(poll)

    def _print_status(self, ts, row, htf_bull, htf2_bull, cross_up, cross_down):
        total_pnl = self.equity - INITIAL_CAPITAL
        n_trades  = len(self.trade_log)
        win_count = sum(1 for t in self.trade_log if t['result'] == 'WIN')
        win_rate  = (win_count / n_trades * 100) if n_trades else 0.0
        unreal    = self._unrealised_pct(row['close']) if self.position else 0.0
        htf2_str  = f"  HTF2({HTF2}): {'BULL' if htf2_bull else 'BEAR'}" if USE_HTF2 and htf2_bull is not None else ""
        cross_str = 'UP' if cross_up else ('DOWN' if cross_down else 'NONE')
        print("-" * 70)
        print(f"[PAPER] Candle #{self.iteration} @ {ts.strftime('%Y-%m-%d %H:%M:%S')}  Price {row['close']:.2f}")
        print(f"EMA{EMA_FAST} {row['ema_fast']:.2f}  EMA{EMA_SLOW} {row['ema_slow']:.2f}  ADX {row['adx']:.2f}  ATR {row['atr']:.2f}")
        print(f"HTF({HTF}): {'BULL' if htf_bull else 'BEAR'}{htf2_str}  Cross: {cross_str}")
        if self.position:
            print(f"Open {self.position}  Entry {self.entry_price:.2f}  SL {self.sl:.2f}  TP {self.tp:.2f}  Unrl {unreal:+.2f}%")
        else:
            print("Position: None")
        print(f"Equity ${self.equity:,.2f}  PnL {total_pnl:+.2f}  Trades {n_trades}  WinRate {win_rate:.1f}%")


# ─────────────────────────────────────────────────────────────────────────────

class LiveTrader:
    """
    Real on-chain execution via DriftExecutor.
    Polls for signals and submits transactions to Drift Protocol.
    """

    def __init__(self):
        self.executor   = DriftExecutor()
        self.position   = None   # 'LONG' | 'SHORT' | None — local state
        self.entry_price= None
        self.sl         = None
        self.tp         = None
        self.trade_log  = []
        self.iteration  = 0

    def _fetch_all(self):
        if DATA_PULL_MODE == 'bars':
            ltf_raw  = get_data(SYMBOL, LTF,  bars=LTF_BARS)
            htf_raw  = get_data(SYMBOL, HTF,  bars=HTF_BARS)
            htf2_raw = get_data(SYMBOL, HTF2, bars=HTF2_BARS) if USE_HTF2 else None
        else:
            ltf_raw  = get_data(SYMBOL, LTF,  days=PAPER_LTF_DAYS)
            htf_raw  = get_data(SYMBOL, HTF,  days=PAPER_HTF_DAYS)
            htf2_raw = get_data(SYMBOL, HTF2, days=PAPER_HTF2_DAYS) if USE_HTF2 else None
        return ltf_raw, htf_raw, htf2_raw

    def _sync_position_with_chain(self):
        """
        Check on-chain position. If Drift closed it (SL/TP hit) but our
        local state still shows open, sync and log the result.
        """
        on_chain = self.executor.get_open_position()
        if self.position and (on_chain is None or on_chain.base_asset_amount == 0):
            log.info("Position closed on-chain (SL or TP hit). Syncing local state.")
            balance = self.executor.get_balance()
            self.trade_log.append({
                'direction'  : self.position,
                'entry_price': self.entry_price,
                'sl': self.sl, 'tp': self.tp,
                'closed_by'  : 'SL/TP on-chain',
                'balance_after': balance,
            })
            self.position    = None
            self.entry_price = self.sl = self.tp = None

    def run(self):
        poll      = PAPER_POLL_INTERVAL
        htf2_info = f" + {HTF2}" if USE_HTF2 else ""
        balance   = self.executor.get_balance()
        print(
            f"\nLive trader — Drift {DRIFT_ENV.upper()}  BTC-PERP  {LTF}/{HTF}{htf2_info}\n"
            f"Drift USDC balance: ${balance:,.2f}  Risk/Trade: {RISK_PCT}%  Leverage: {DRIFT_LEVERAGE}x\n"
            f"Poll: {poll}s  Ctrl+C to stop\n"
        )

        while True:
            try:
                self.iteration += 1

                # ── Sync position state with chain ────────────────────────────
                self._sync_position_with_chain()

                # ── Fetch market data & compute signals ───────────────────────
                ltf_raw, htf_raw, htf2_raw = self._fetch_all()
                ltf  = compute_indicators(ltf_raw)
                htf  = compute_indicators(htf_raw)
                htf2 = compute_indicators(htf2_raw) if USE_HTF2 and htf2_raw is not None else None

                row, prev = ltf.iloc[-1], ltf.iloc[-2]
                ts        = ltf.index[-1]
                htf_bull  = htf.iloc[-1]['ema_fast'] > htf.iloc[-1]['ema_slow']
                htf2_bull = (htf2.iloc[-1]['ema_fast'] > htf2.iloc[-1]['ema_slow']) if htf2 is not None else None

                cross_up   = (row['ema_fast'] > row['ema_slow']) and (prev['ema_fast'] <= prev['ema_slow'])
                cross_down = (row['ema_fast'] < row['ema_slow']) and (prev['ema_fast'] >= prev['ema_slow'])
                adx_ok     = row['adx'] > ADX_THRESHOLD
                atr_ok     = row['atr'] > row['atr_ma']

                if USE_HTF2 and htf2_bull is not None:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull  and htf2_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull) and (not htf2_bull)
                else:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull)

                # ── Execute if flat and signal fired ──────────────────────────
                if self.position is None:
                    if long_ok or short_ok:
                        sgn = 1 if long_ok else -1
                        sl_price = row['close'] - sgn * ATR_SL_MULT * row['atr']
                        tp_price = row['close'] + sgn * ATR_TP_MULT * row['atr']
                        if long_ok:
                            tx = self.executor.open_long(sl_price, tp_price)
                            self.position = 'LONG'
                        else:
                            tx = self.executor.open_short(sl_price, tp_price)
                            self.position = 'SHORT'
                        self.entry_price = row['close']
                        self.sl, self.tp = sl_price, tp_price
                        log.info(f"Trade opened: {self.position}  entry={self.entry_price:.2f}  "
                                 f"SL={sl_price:.2f}  TP={tp_price:.2f}  tx={tx}")

                self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)

            except KeyboardInterrupt:
                print("\nStopped by user.")
                # Optionally close any open position on exit:
                # if self.position:
                #     self.executor.close_position()
                break
            except Exception as e:
                log.error(f"Loop error: {e}")
            finally:
                pass
            time.sleep(poll)

        self.executor.shutdown()

    def _print_status(self, ts, row, htf_bull, htf2_bull, cross_up, cross_down):
        n_trades  = len(self.trade_log)
        htf2_str  = f"  HTF2({HTF2}): {'BULL' if htf2_bull else 'BEAR'}" if USE_HTF2 and htf2_bull is not None else ""
        cross_str = 'UP' if cross_up else ('DOWN' if cross_down else 'NONE')
        balance   = self.executor.get_balance()
        print("-" * 70)
        print(f"[LIVE] Candle #{self.iteration} @ {ts.strftime('%Y-%m-%d %H:%M:%S')}  Price {row['close']:.2f}")
        print(f"EMA{EMA_FAST} {row['ema_fast']:.2f}  EMA{EMA_SLOW} {row['ema_slow']:.2f}  ADX {row['adx']:.2f}  ATR {row['atr']:.2f}")
        print(f"HTF({HTF}): {'BULL' if htf_bull else 'BEAR'}{htf2_str}  Cross: {cross_str}")
        if self.position:
            print(f"On-chain {self.position}  Entry {self.entry_price:.2f}  SL {self.sl:.2f}  TP {self.tp:.2f}")
        else:
            print("Position: None")
        print(f"Drift USDC balance: ${balance:,.2f}  Completed trades: {n_trades}")

## 9) Runner

Set `MODE` in the Config cell to choose what runs:

| MODE | What happens |
|---|---|
| `backtest` | Historical simulation on CCXT data — no wallet needed |
| `paper` | Live signals, simulated fills printed to console — no wallet needed |
| `live` | **Real transactions on Drift Protocol** — wallet + RPC required |

In [ ]:
from math import sqrt

def run_backtest():
    htf2_info = f" + {HTF2}" if USE_HTF2 else ""
    print(f"Fetching {SYMBOL}  LTF:{LTF}  HTF:{HTF}{htf2_info}")

    if DATA_PULL_MODE == 'bars':
        ltf_raw  = get_data(SYMBOL, LTF,  bars=LTF_BARS)
        htf_raw  = get_data(SYMBOL, HTF,  bars=HTF_BARS)
        htf2_raw = get_data(SYMBOL, HTF2, bars=HTF2_BARS) if USE_HTF2 else None
    else:
        ltf_raw  = get_data(SYMBOL, LTF,  days=BACKTEST_DAYS)
        htf_raw  = get_data(SYMBOL, HTF,  days=max(BACKTEST_DAYS * 2, BACKTEST_DAYS + 7))
        htf2_raw = get_data(SYMBOL, HTF2, days=max(BACKTEST_DAYS * 4, BACKTEST_DAYS + 30)) if USE_HTF2 else None

    ltf  = compute_indicators(ltf_raw)
    htf  = compute_indicators(htf_raw)
    htf2 = compute_indicators(htf2_raw) if USE_HTF2 and htf2_raw is not None else None

    df    = generate_signals(ltf, htf, htf2)
    bt    = Backtester(df)
    stats = bt.run()

    if 'error' in stats:
        print('WARNING:', stats['error'])
        return

    tf_label    = f"{LTF} -> {HTF}" + (f" -> {HTF2}" if USE_HTF2 else "")
    htf2_status = f"ON ({HTF2})" if USE_HTF2 else "OFF"

    print("\n" + "=" * 42)
    print("BACKTEST RESULTS  [Drift Bot]")
    print("=" * 42)
    print(f"Symbol         : {SYMBOL}")
    print(f"Timeframes     : {tf_label}")
    print(f"HTF2 filter    : {htf2_status}")
    print(f"Total trades   : {stats['total_trades']}")
    print(f"Win rate       : {stats['win_rate']}%")
    print(f"Total return   : {stats['total_return']}%")
    print(f"Final equity   : ${stats['final_equity']}")
    print(f"Sharpe ratio   : {stats['sharpe_ratio']}")
    print(f"Max drawdown   : {stats['max_drawdown']}%")
    print(f"Profit factor  : {stats['profit_factor']}")
    print(f"Avg win        : ${stats['avg_win']}")
    print(f"Avg loss       : ${stats['avg_loss']}")
    print("=" * 42 + "\n")
    print(stats['trades_df'].to_string(index=False))


# ── Dispatch ──────────────────────────────────────────────────────────────────
if MODE == 'backtest':
    run_backtest()
elif MODE == 'paper':
    PaperTrader().run()
elif MODE == 'live':
    LiveTrader().run()
else:
    print(f"Unknown MODE='{MODE}'. Use: backtest | paper | live")